# 3. Evaluation & Visualization
## MVTec Metal Nut - Binary Classification

Load trained model, evaluate on test set, visualize results and predictions

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
import pickle
import json
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Set random seed
torch.manual_seed(43)
np.random.seed(43)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Load Model & Data

In [ ]:
# Load model info and training history
with open('../models/model_info.json', 'r') as f:
    model_info = json.load(f)

with open('../models/training_history.pkl', 'rb') as f:
    history = pickle.load(f)

# Recreate model architecture
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

# Load trained weights
model.load_state_dict(torch.load('../models/resnet18_mvtec_binary.pth', map_location=device))
model = model.to(device)
model.eval()

print("✅ Model loaded successfully")
print(f"   Architecture: {model_info['architecture']}")
print(f"   Trained for {len(history['train_loss'])} epochs")

# Load data
data_config = pickle.load(open('../data/dataloaders.pkl', 'rb'))
test_loader = data_config['test_loader']
imagenet_mean = model_info['normalization']['mean']
imagenet_std = model_info['normalization']['std']

print(f"✅ Test data loaded ({len(test_loader)} batches)")

## Test Set Evaluation

In [ ]:
criterion = nn.CrossEntropyLoss()
all_preds = []
all_labels = []
all_probs = []
test_loss = 0.0

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        loss = criterion(outputs, y)
        test_loss += loss.item() * X.size(0)
        
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_loss /= len(all_labels)
test_acc = np.mean(np.array(all_preds) == np.array(all_labels))

print("="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({int(test_acc*len(all_labels))}/{len(all_labels)})")
print(f"\n{classification_report(all_labels, all_preds, target_names=['Good', 'Defective'])}")

## Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o', markersize=4, linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', markersize=4, linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o', markersize=4, linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s', markersize=4, linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].set_title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training history plot saved to: ../models/training_history.png")

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Test Set', fontsize=12, fontweight='bold')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Good', 'Defective'], fontsize=10)
plt.yticks(tick_marks, ['Good', 'Defective'], fontsize=10)

# Add text annotations
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', 
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14, fontweight='bold')

plt.ylabel('True Label', fontsize=11)
plt.xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved to: ../models/confusion_matrix.png")

## Sample Predictions

Visualize predictions on random test samples

In [ ]:
# Function to denormalize image
def denormalize(img, mean, std):
    img = img.clone()
    for i in range(3):
        img[i] = img[i] * std[i] + mean[i]
    return img.clamp(0, 1)

# Get a batch to visualize
sample_batch = next(iter(test_loader))
images, labels = sample_batch

with torch.no_grad():
    outputs = model(images.to(device))
    probs = torch.softmax(outputs, dim=1)
    preds = outputs.argmax(dim=1)

# Show 12 samples
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

class_names = ['Good', 'Defective']
colors = ['green', 'red']

for idx in range(min(12, len(images))):
    ax = axes[idx]
    
    # Denormalize and show image
    img = denormalize(images[idx], imagenet_mean, imagenet_std)
    img = img.permute(1, 2, 0).numpy()
    ax.imshow(img)
    
    true_label = class_names[labels[idx].item()]
    pred_label = class_names[preds[idx].item()]
    confidence = probs[idx, preds[idx]].item()
    
    # Color title based on correctness
    color = 'green' if labels[idx] == preds[idx] else 'red'
    title = f"True: {true_label}\nPred: {pred_label}\nConf: {confidence:.2f}"
    ax.set_title(title, fontsize=10, fontweight='bold', color=color)
    ax.axis('off')

plt.suptitle('Sample Predictions on Test Set', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../models/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Sample predictions saved to: ../models/sample_predictions.png")

## Summary & Next Steps

All evaluation files have been saved to the `../models/` directory for easy access and sharing.

In [ ]:
print("="*70)
print("✅ EVALUATION COMPLETE")
print("="*70)
print("\nSaved artifacts:")
print("  📊 ../models/training_history.png     - Loss & accuracy curves")
print("  📊 ../models/confusion_matrix.png     - Test set confusion matrix")
print("  📊 ../models/sample_predictions.png   - Sample predictions")
print("  💾 ../models/resnet18_mvtec_binary.pth - Trained model weights")
print("  📄 ../models/model_info.json          - Model metadata")
print("\nReady to:")
print("  - Push to GitHub with all visualization and model files")
print("  - Transfer to Colab and continue training with different settings")
print("  - Try Grad-CAM or other interpretability techniques")
print("="*70)